# **Data Preparation and Pre-processing**




*   ### Data from the 4 classes, namely, Infill, Solid, Wire Good, and Wire Bad is imported and processed.
*   ### Data is augmented by applying segmentation and rotation.
*   ### Output= Augmented data frames of all classes (1 File= 36 Files).






> Necessary libraries are imported



In [ ]:
import pandas as pd
import numpy as np
import os,re

### **Importing Data for Infill Class**



> Data is being imported from drive



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

> Configure paths and constants before running any cells below.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────
# Adjust DATA_ROOT and OUTPUT_ROOT to match your local or Drive structure.
# When running on Google Colab, set DATA_ROOT inside your mounted Drive.

DATA_ROOT   = "/content/drive/MyDrive/Hanan Share"   # raw scan CSVs
OUTPUT_ROOT = "/content/drive/MyDrive/Hanan Share/Data_augmentation"  # augmented output

# LLS height thresholds (mm)
HEIGHT_MIN = 1
HEIGHT_MAX = 40

# Target image size (pixels)
IMG_SIZE = 224


> Segment coordinates and data-loading helper shared by all classes.

In [ ]:
# Sliding-window segment coordinates (row, col) – 9 overlapping 224×224 crops
# from a ~600×600 scan frame. Shared across all classes.
SEGMENT_COORDINATES = [
    [(1, 1),   (1, 224),   (224, 1),   (224, 224)],
    [(1, 188), (1, 411),   (224, 188), (224, 411)],
    [(1, 371), (1, 594),   (224, 371), (224, 594)],
    [(188, 1), (188, 224), (411, 1),   (411, 224)],
    [(188, 188),(188, 411),(411, 188), (411, 411)],
    [(188, 371),(188, 594),(411, 371), (411, 594)],
    [(376, 1), (376, 224), (599, 1),   (599, 224)],
    [(376, 188),(376, 411),(600, 187), (599, 411)],
    [(376, 372),(376, 595),(599, 371), (599, 595)],
]


In [ ]:
def load_and_clean(class_dir):
    """Read all CSVs from *class_dir*, clip outliers, and return a dict of DataFrames."""
    files = [f for f in os.listdir(class_dir) if f.endswith('.csv')]
    dataframes = {}
    for fname in files:
        data = pd.read_csv(os.path.join(class_dir, fname), header=None)
        data = data.map(lambda x: np.nan if x < HEIGHT_MIN or x > HEIGHT_MAX else x)
        avg = (data.unstack().nsmallest().iloc[0] + data.unstack().nlargest().iloc[0]) / 2
        data = data.fillna(avg)
        dataframes[fname] = data
    return dataframes


### **Importing Data for Infill Class**

In [ ]:
dataframes = load_and_clean(os.path.join(DATA_ROOT, "Infill"))
print(f"Infill files loaded: {len(dataframes)}")


# **Data Augmentation – Infill**

> Segmentation Procedure

In [ ]:
output_folder = os.path.join(OUTPUT_ROOT, "Infill")
os.makedirs(output_folder, exist_ok=True)

for file_name, data in dataframes.items():
    for i, coords in enumerate(SEGMENT_COORDINATES, 1):
        start_row, start_col = coords[0]
        end_row,   end_col   = coords[3]
        segment_data = data.iloc[start_row - 1:end_row, start_col - 1:end_col]
        seg_fname = f"{os.path.splitext(file_name)[0]}_segment_{i}_Infill.csv"
        segment_data.to_csv(os.path.join(output_folder, seg_fname), index=False, header=False)


> Rotation Procedure

In [ ]:
input_folder  = os.path.join(OUTPUT_ROOT, "Infill")
output_folder = os.path.join(OUTPUT_ROOT, "Infill")
os.makedirs(output_folder, exist_ok=True)

def rotate_90_degrees(matrix):
    return np.rot90(matrix, k=1)

for file_name in os.listdir(input_folder):
    if not file_name.endswith('.csv'):
        continue
    original_data = pd.read_csv(os.path.join(input_folder, file_name), header=None)
    rotated_90  = rotate_90_degrees(original_data)
    rotated_180 = rotate_90_degrees(rotated_90)
    rotated_270 = rotate_90_degrees(rotated_180)
    base = os.path.splitext(file_name)[0]
    for angle, rot in [(90, rotated_90), (180, rotated_180), (270, rotated_270)]:
        out_name = f"{base}_rotated_{angle}_Infill.csv"
        pd.DataFrame(rot).to_csv(os.path.join(output_folder, out_name), index=False, header=False)


### **Importing data for Solid Class**

In [ ]:
dataframes = load_and_clean(os.path.join(DATA_ROOT, "Solid"))
print(f"Solid files loaded: {len(dataframes)}")


# **Data Augmentation – Solid**

> Segmentation Procedure

In [ ]:
output_folder = os.path.join(OUTPUT_ROOT, "Solid")
os.makedirs(output_folder, exist_ok=True)

for file_name, data in dataframes.items():
    for i, coords in enumerate(SEGMENT_COORDINATES, 1):
        start_row, start_col = coords[0]
        end_row,   end_col   = coords[3]
        segment_data = data.iloc[start_row - 1:end_row, start_col - 1:end_col]
        seg_fname = f"{os.path.splitext(file_name)[0]}_segment_{i}_solid.csv"
        segment_data.to_csv(os.path.join(output_folder, seg_fname), index=False, header=False)


> Rotation Procedure

In [ ]:
input_folder  = os.path.join(OUTPUT_ROOT, "Solid")
output_folder = os.path.join(OUTPUT_ROOT, "Solid")
os.makedirs(output_folder, exist_ok=True)

def rotate_90_degrees(matrix):
    return np.rot90(matrix, k=1)

for file_name in os.listdir(input_folder):
    if not file_name.endswith('.csv'):
        continue
    original_data = pd.read_csv(os.path.join(input_folder, file_name), header=None)
    rotated_90  = rotate_90_degrees(original_data)
    rotated_180 = rotate_90_degrees(rotated_90)
    rotated_270 = rotate_90_degrees(rotated_180)
    base = os.path.splitext(file_name)[0]
    for angle, rot in [(90, rotated_90), (180, rotated_180), (270, rotated_270)]:
        out_name = f"{base}_rotated_{angle}_solid.csv"
        pd.DataFrame(rot).to_csv(os.path.join(output_folder, out_name), index=False, header=False)


### **Importing data for Wire Good Class**

In [ ]:
dataframes = load_and_clean(os.path.join(DATA_ROOT, "Draht_good_round"))
print(f"Wire Good files loaded: {len(dataframes)}")


# **Data Augmentation – Wire Good**

> Segmentation Procedure

In [ ]:
output_folder = os.path.join(OUTPUT_ROOT, "Wire_good")
os.makedirs(output_folder, exist_ok=True)

for file_name, data in dataframes.items():
    for i, coords in enumerate(SEGMENT_COORDINATES, 1):
        start_row, start_col = coords[0]
        end_row,   end_col   = coords[3]
        segment_data = data.iloc[start_row - 1:end_row, start_col - 1:end_col]
        seg_fname = f"{os.path.splitext(file_name)[0]}_segment_{i}_Wire_good.csv"
        segment_data.to_csv(os.path.join(output_folder, seg_fname), index=False, header=False)


> Rotation Procedure

In [ ]:
input_folder  = os.path.join(OUTPUT_ROOT, "Wire_good")
output_folder = os.path.join(OUTPUT_ROOT, "Wire_good")
os.makedirs(output_folder, exist_ok=True)

def rotate_90_degrees(matrix):
    return np.rot90(matrix, k=1)

for file_name in os.listdir(input_folder):
    if not file_name.endswith('.csv'):
        continue
    original_data = pd.read_csv(os.path.join(input_folder, file_name), header=None)
    rotated_90  = rotate_90_degrees(original_data)
    rotated_180 = rotate_90_degrees(rotated_90)
    rotated_270 = rotate_90_degrees(rotated_180)
    base = os.path.splitext(file_name)[0]
    for angle, rot in [(90, rotated_90), (180, rotated_180), (270, rotated_270)]:
        out_name = f"{base}_rotated_{angle}_Wire_good.csv"
        pd.DataFrame(rot).to_csv(os.path.join(output_folder, out_name), index=False, header=False)


### **Importing data for Wire Bad Class**

In [ ]:
dataframes = load_and_clean(os.path.join(DATA_ROOT, "Wire_bad"))
print(f"Wire Bad files loaded: {len(dataframes)}")


# **Data Augmentation – Wire Bad**

> Segmentation Procedure

In [ ]:
output_folder = os.path.join(OUTPUT_ROOT, "Wire_bad")
os.makedirs(output_folder, exist_ok=True)

for file_name, data in dataframes.items():
    for i, coords in enumerate(SEGMENT_COORDINATES, 1):
        start_row, start_col = coords[0]
        end_row,   end_col   = coords[3]
        segment_data = data.iloc[start_row - 1:end_row, start_col - 1:end_col]
        seg_fname = f"{os.path.splitext(file_name)[0]}_segment_{i}_Wire_bad.csv"
        segment_data.to_csv(os.path.join(output_folder, seg_fname), index=False, header=False)


> Rotation Procedure

In [ ]:
input_folder  = os.path.join(OUTPUT_ROOT, "Wire_bad")
output_folder = os.path.join(OUTPUT_ROOT, "Wire_bad")
os.makedirs(output_folder, exist_ok=True)

def rotate_90_degrees(matrix):
    return np.rot90(matrix, k=1)

for file_name in os.listdir(input_folder):
    if not file_name.endswith('.csv'):
        continue
    original_data = pd.read_csv(os.path.join(input_folder, file_name), header=None)
    rotated_90  = rotate_90_degrees(original_data)
    rotated_180 = rotate_90_degrees(rotated_90)
    rotated_270 = rotate_90_degrees(rotated_180)
    base = os.path.splitext(file_name)[0]
    for angle, rot in [(90, rotated_90), (180, rotated_180), (270, rotated_270)]:
        out_name = f"{base}_rotated_{angle}_Wire_bad.csv"
        pd.DataFrame(rot).to_csv(os.path.join(output_folder, out_name), index=False, header=False)
